# Task

Prepare an end-to-end ETL with PySpark to

* extract data from the [kaggle spotify dataset](https://www.kaggle.com/datasets/kapturovalexander/spotify-data-from-pyspark-course/data)
* transform it (cleaning, calculate KPIs)
* and load results into local files

In [111]:
!pip install kagglehub

In [1]:
import kagglehub

# Download latest version
path_spotify_data = kagglehub.dataset_download(
    "kapturovalexander/spotify-data-from-pyspark-course"
)

print("Path to dataset files:", path_spotify_data)

Path to dataset files: /home/jovyan/.cache/kagglehub/datasets/kapturovalexander/spotify-data-from-pyspark-course/versions/53


## Extract

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import Window
import pyspark.sql.functions as sf
f = sf

In [3]:
spark = SparkSession.builder.appName("Spark-ETL-pipeline").getOrCreate()

In [33]:
#help(spark.read.csv)

In [4]:
#df_spotify = df = spark.read.csv("data/spotify-data.csv", header=True, inferSchema=True)
df_spotify_inferred = df = (
    spark.read

    # alternative options only for testing
    .option("delimiter", ",")  # field delimiter, default: ','
    .option("sep", ",")        # field delimiter, default: ','
    
    .csv(
        path_spotify_data,
        header=True,
        inferSchema=True,
        sep=r',',          # only 'sep' works as parameter to csv(). 'delimiter' is not allowed (default: ',')
        quote=r'"',        # character used to quote any string which contains quote characters (default: '"')
        escape=r'"',       # unescape quotes within a quoted string escaped by doubling them as "" (default: '\')
        encoding="UTF-8",  # default: utf-8
    )
)

In [7]:
df.take(5)

[Row(id='6KbQ3uYMLKb5jDxLF7wYDD', name='Singende Bataillone 1. Teil', artists="['Carl Woitschach']", duration_ms=158648, release_date='1928', year=1928, acousticness=0.995, danceability=0.708, energy=0.195, instrumentalness=0.563, liveness=0.151, loudness=-12.428, speechiness=0.0506, tempo=118.469, valence=0.779, mode=1, key=10, popularity=0, explicit=0),
 Row(id='6KuQTIu1KoTTkLXKrwlLPV', name='Fantasiestücke, Op. 111: Più tosto lento', artists="['Robert Schumann', 'Vladimir Horowitz']", duration_ms=282133, release_date='1928', year=1928, acousticness=0.994, danceability=0.379, energy=0.0135, instrumentalness=0.901, liveness=0.0763, loudness=-28.454, speechiness=0.0462, tempo=83.972, valence=0.0767, mode=1, key=8, popularity=0, explicit=0),
 Row(id='6L63VW0PibdM1HDSBoqnoM', name='Chapter 1.18 - Zamek kaniowski', artists="['Seweryn Goszczyński']", duration_ms=104300, release_date='1928', year=1928, acousticness=0.604, danceability=0.749, energy=0.22, instrumentalness=0.0, liveness=0.119

In [6]:
print(f"Number of rows: {df.count()}")

print("\nInferred Schema:")
df.printSchema()

print("\nPreview:")
df.show(5)

Number of rows: 169909

Inferred Schema:
root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- release_date: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- key: integer (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- explicit: integer (nullable = true)


Preview:
+--------------------+--------------------+--------------------+-----------+------------+----+------------+------------+------+----------------+--------+--------+-----------+-----

### Specifying the schema

[Data types reference](https://spark.apache.org/docs/latest/sql-ref-datatypes.html)

In [9]:
df.columns

['id',
 'name',
 'artists',
 'duration_ms',
 'release_date',
 'year',
 'acousticness',
 'danceability',
 'energy',
 'instrumentalness',
 'liveness',
 'loudness',
 'speechiness',
 'tempo',
 'valence',
 'mode',
 'key',
 'popularity',
 'explicit']

+--------+
|explicit|
+--------+
|       1|
|       0|
+--------+



In [48]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    FloatType,
    DoubleType,  # double precision float, not required here
    DateType,
    BooleanType,
)

schema = StructType([
    StructField("id", StringType(), nullable=False),
    
    StructField("name", StringType(), nullable=True),
    StructField("artists", StringType(), nullable=True),
    StructField("duration_ms", IntegerType(), nullable=True),
    
    StructField("release_date", StringType(), nullable=True),  # needs cleaning
    #StructField("release_date", DateType(), nullable=True),
    
    StructField("year", IntegerType(), nullable=True),
    StructField("acousticness", FloatType(), nullable=True),
    StructField("danceability", FloatType(), nullable=True),
    StructField("energy", FloatType(), nullable=True),
    StructField("instrumentalness", FloatType(), nullable=True),
    StructField("liveness", FloatType(), nullable=True),
    StructField("loudness", FloatType(), nullable=True),
    StructField("speechiness", FloatType(), nullable=True),
    StructField("tempo", FloatType(), nullable=True),
    StructField("valence", FloatType(), nullable=True),
    StructField("mode", IntegerType(), nullable=True),
    StructField("key", IntegerType(), nullable=True),
    StructField("popularity", FloatType(), nullable=True),
    StructField("explicit", BooleanType(), nullable=True),
])

In [49]:
df_spotify = df = (
    spark.read
    .csv(
        path_spotify_data,
        schema,
        header=True,
        sep=r',',          # only 'sep' works as parameter to csv(). 'delimiter' is not allowed (default: ',')
        quote=r'"',        # character used to quote any string which contains quote characters (default: '"')
        escape=r'"',       # unescape quotes within a quoted string escaped by doubling them as "" (default: '\')
        encoding="UTF-8",  # default: utf-8
    )
)

In [29]:
df_spotify.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- release_date: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- acousticness: float (nullable = true)
 |-- danceability: float (nullable = true)
 |-- energy: float (nullable = true)
 |-- instrumentalness: float (nullable = true)
 |-- liveness: float (nullable = true)
 |-- loudness: float (nullable = true)
 |-- speechiness: float (nullable = true)
 |-- tempo: float (nullable = true)
 |-- valence: float (nullable = true)
 |-- mode: integer (nullable = true)
 |-- key: integer (nullable = true)
 |-- popularity: float (nullable = true)
 |-- explicit: integer (nullable = true)



In [50]:
res = (
    df_spotify
    .dropDuplicates()
    .dropna(subset=["id", "name", "artists", "year", "popularity", ])
    
)
res.count()

# all rows = 169909

169909